# Pipeline de Ingesta RAG — Project GenAI

Este notebook genera el indice FAISS para el pipeline RAG.

**Arquitectura:** GCS (PDFs) → PyPDF → Fragmentos → Embeddings (text-embedding-004) → FAISS → GCS

**Por que Colab?** El Docker image del proyecto incluye `google-cloud-aiplatform` (~4-6GB RAM al importar). Este notebook usa solo dependencias ligeras (~200MB total).

**RAM necesaria:** ~2-3 GB (Colab Free tiene ~12 GB — mas que suficiente)

In [ ]:
# ============================================================
# Celda 1: Instalar dependencias minimas
# ============================================================
# Solo lo necesario — sin LangChain, sin google-cloud-aiplatform
# Esto consume ~200 MB vs ~6 GB del stack completo

!pip install -q google-generativeai google-cloud-storage faiss-cpu pypdf numpy

print("Dependencias instaladas OK")

In [ ]:
# ============================================================
# Celda 2: Configuracion y autenticacion
# ============================================================

# --- CONFIGURAR ESTOS VALORES ---
GOOGLE_API_KEY = ""  # Tu API key de Google AI Studio
GCS_BUCKET_NAME = ""  # Nombre de tu bucket GCS (ej: genai-docs-mi-proyecto)
GCS_PDFS_PREFIX = "pdfs/"  # Prefijo donde estan los PDFs en el bucket
GCS_VECTORSTORE_PREFIX = "vectorstore/"  # Prefijo donde se guardara el indice

# Parametros RAG
CHUNK_SIZE = 500
CHUNK_OVERLAP = 100
EMBEDDING_MODEL = "models/text-embedding-004"
EMBEDDING_DIM = 768
BATCH_SIZE = 5  # Fragmentos por llamada a la API
PAUSE_BETWEEN_BATCHES = 2  # Segundos entre lotes

# --- Validar ---
assert GOOGLE_API_KEY, "Falta GOOGLE_API_KEY"
assert GCS_BUCKET_NAME, "Falta GCS_BUCKET_NAME"

# --- Autenticar con GCP para acceder a GCS ---
from google.colab import auth
auth.authenticate_user()
print("Autenticado con GCP OK")

# --- Configurar API key de Gemini ---
import google.generativeai as genai
genai.configure(api_key=GOOGLE_API_KEY)
print("Google AI Studio configurado OK")

# --- Conectar a GCS ---
from google.cloud import storage
gcs_client = storage.Client()
bucket = gcs_client.bucket(GCS_BUCKET_NAME)
print(f"Bucket: gs://{GCS_BUCKET_NAME}")

# Listar PDFs disponibles
blobs = list(bucket.list_blobs(prefix=GCS_PDFS_PREFIX))
pdf_blobs = [b for b in blobs if b.name.endswith(".pdf")]
print(f"PDFs encontrados: {len(pdf_blobs)}")
for b in pdf_blobs:
    print(f"  {b.name} ({b.size / 1024:.0f} KB)")

In [ ]:
# ============================================================
# Celda 3: Descargar PDFs, fragmentar y generar embeddings
# ============================================================

import gc
import os
import re
import time
import tempfile
import numpy as np
import faiss
from pypdf import PdfReader


def clasificar_pdf(nombre):
    """Clasifica un PDF por su nombre en categoria."""
    nombre_lower = nombre.lower()
    if any(kw in nombre_lower for kw in ["politica", "política", "codigo", "código"]):
        return "POLITICAS"
    elif "procedimiento" in nombre_lower:
        return "PROCEDIMIENTOS"
    elif "reglamento" in nombre_lower:
        return "REGLAMENTOS"
    return "OTROS"


def fragmentar_texto(texto, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    """Divide texto en fragmentos con solapamiento."""
    fragmentos = []
    inicio = 0
    while inicio < len(texto):
        fin = inicio + chunk_size
        fragmento = texto[inicio:fin].strip()
        if fragmento:
            fragmentos.append(fragmento)
        inicio += chunk_size - overlap
    return fragmentos


def generar_embeddings(textos, max_retries=7):
    """Genera embeddings con reintentos para rate limits."""
    for intento in range(max_retries):
        try:
            vectors = []
            for texto in textos:
                result = genai.embed_content(
                    model=EMBEDDING_MODEL,
                    content=texto
                )
                vectors.append(result["embedding"])
            return vectors
        except Exception as e:
            if "429" in str(e) or "RESOURCE_EXHAUSTED" in str(e):
                espera = min(2 ** (intento + 1), 60)
                print(f"    Rate limit, reintentando en {espera}s... ({intento+1}/{max_retries})")
                time.sleep(espera)
            else:
                raise
    print(f"    ERROR: No se pudo generar embeddings tras {max_retries} intentos")
    return None


# --- Estructuras para el indice ---
all_vectors = []  # Lista de arrays numpy
all_metadata = []  # Lista de dicts {texto, fuente, pagina, categoria}

# --- Procesar cada PDF ---
inicio_total = time.time()
total_fragmentos = 0
categorias = {}

for idx, blob in enumerate(pdf_blobs, 1):
    nombre_pdf = os.path.basename(blob.name)
    categoria = clasificar_pdf(nombre_pdf)

    # Descargar PDF a /tmp
    tmp_path = os.path.join(tempfile.gettempdir(), nombre_pdf)
    blob.download_to_filename(tmp_path)

    # Extraer texto pagina por pagina
    try:
        reader = PdfReader(tmp_path)
    except Exception as e:
        print(f"  [{idx}/{len(pdf_blobs)}] {nombre_pdf}: ERROR leyendo PDF - {e}")
        continue

    fragmentos_pdf = []
    for num_pagina, pagina in enumerate(reader.pages, 1):
        texto = pagina.extract_text() or ""
        texto = texto.strip()
        if not texto:
            continue
        chunks = fragmentar_texto(texto)
        for chunk in chunks:
            fragmentos_pdf.append({
                "texto": chunk,
                "fuente": nombre_pdf,
                "pagina": num_pagina,
                "categoria": categoria,
            })

    if not fragmentos_pdf:
        print(f"  [{idx}/{len(pdf_blobs)}] {nombre_pdf}: 0 fragmentos (vacio)")
        os.remove(tmp_path)
        continue

    # Generar embeddings en lotes
    for i in range(0, len(fragmentos_pdf), BATCH_SIZE):
        lote = fragmentos_pdf[i:i + BATCH_SIZE]
        textos = [f["texto"] for f in lote]
        vectors = generar_embeddings(textos)

        if vectors is None:
            continue

        all_vectors.extend(vectors)
        all_metadata.extend(lote)

        if i + BATCH_SIZE < len(fragmentos_pdf):
            time.sleep(PAUSE_BETWEEN_BATCHES)

    total_fragmentos += len(fragmentos_pdf)
    categorias[categoria] = categorias.get(categoria, 0) + len(fragmentos_pdf)
    print(f"  [{idx}/{len(pdf_blobs)}] {nombre_pdf}: {len(fragmentos_pdf)} fragmentos — {categoria}")

    # Limpiar
    os.remove(tmp_path)
    del reader, fragmentos_pdf
    gc.collect()

    # Pausa entre PDFs para rate limits
    time.sleep(1)

duracion = time.time() - inicio_total
print(f"\n{'='*60}")
print(f"Fragmentacion y embeddings completados")
print(f"  PDFs procesados:    {len(pdf_blobs)}")
print(f"  Fragmentos totales: {total_fragmentos}")
print(f"  Vectores generados: {len(all_vectors)}")
print(f"  Categorias:         {categorias}")
print(f"  Tiempo:             {duracion:.0f} segundos")
print(f"{'='*60}")

In [ ]:
# ============================================================
# Celda 4: Construir indice FAISS y subir a GCS
# ============================================================

import pickle
import uuid

assert len(all_vectors) > 0, "No hay vectores para construir el indice"

# --- Construir indice FAISS ---
print(f"Construyendo indice FAISS con {len(all_vectors)} vectores de {EMBEDDING_DIM} dimensiones...")
vectors_np = np.array(all_vectors, dtype=np.float32)
index = faiss.IndexFlatL2(EMBEDDING_DIM)
index.add(vectors_np)
print(f"  Indice FAISS: {index.ntotal} vectores indexados")

# --- Construir docstore compatible con LangChain ---
# Esto permite que rag_search.py cargue el indice con FAISS.load_local()
from langchain_core.documents import Document
from langchain_community.docstore.in_memory import InMemoryDocstore

# Necesitamos instalar langchain para el formato de serializacion
# Pero NO importamos el SDK pesado de embeddings
try:
    from langchain_core.documents import Document
    from langchain_community.docstore.in_memory import InMemoryDocstore
except ImportError:
    print("Instalando langchain-core y langchain-community para formato de serializacion...")
    import subprocess
    subprocess.run(["pip", "install", "-q", "langchain-core", "langchain-community"], check=True)
    from langchain_core.documents import Document
    from langchain_community.docstore.in_memory import InMemoryDocstore

docstore = InMemoryDocstore()
index_to_docstore_id = {}

for i, meta in enumerate(all_metadata):
    doc_id = str(uuid.uuid4())
    doc = Document(
        page_content=meta["texto"],
        metadata={
            "fuente": meta["fuente"],
            "pagina": meta["pagina"],
            "categoria": meta["categoria"],
            "keywords": "",
        }
    )
    docstore.add({doc_id: doc})
    index_to_docstore_id[i] = doc_id

print(f"  Docstore: {len(index_to_docstore_id)} documentos")

# --- Guardar en /tmp ---
output_dir = "/tmp/vectorstore"
os.makedirs(output_dir, exist_ok=True)

# Guardar index.faiss
faiss.write_index(index, os.path.join(output_dir, "index.faiss"))

# Guardar index.pkl (docstore + mapping, formato LangChain)
pkl_data = (docstore, index_to_docstore_id)
with open(os.path.join(output_dir, "index.pkl"), "wb") as f:
    pickle.dump(pkl_data, f)

faiss_size = os.path.getsize(os.path.join(output_dir, "index.faiss")) / 1024
pkl_size = os.path.getsize(os.path.join(output_dir, "index.pkl")) / 1024
print(f"\n  Archivos generados:")
print(f"    index.faiss: {faiss_size:.1f} KB")
print(f"    index.pkl:   {pkl_size:.1f} KB")

# --- Subir a GCS ---
print(f"\nSubiendo a gs://{GCS_BUCKET_NAME}/{GCS_VECTORSTORE_PREFIX}...")
for filename in ["index.faiss", "index.pkl"]:
    local_path = os.path.join(output_dir, filename)
    blob_path = f"{GCS_VECTORSTORE_PREFIX}{filename}"
    blob = bucket.blob(blob_path)
    blob.upload_from_filename(local_path)
    print(f"  Subido: gs://{GCS_BUCKET_NAME}/{blob_path}")

print(f"\n{'='*60}")
print(f"INGESTA COMPLETADA")
print(f"  Vectores en FAISS:     {index.ntotal}")
print(f"  Documentos en store:   {len(index_to_docstore_id)}")
print(f"  Ubicacion GCS:         gs://{GCS_BUCKET_NAME}/{GCS_VECTORSTORE_PREFIX}")
print(f"{'='*60}")
print(f"\nSiguiente paso: el backend de Cloud Run descargara")
print(f"automaticamente este indice de GCS al arrancar.")

In [ ]:
# ============================================================
# Celda 5 (Opcional): Prueba rapida de busqueda semantica
# ============================================================

query = "politica de seguridad de la informacion"
print(f"Query: '{query}'\n")

# Generar embedding de la query
result = genai.embed_content(model=EMBEDDING_MODEL, content=query)
query_vector = np.array([result["embedding"]], dtype=np.float32)

# Buscar los 3 mas similares
scores, indices = index.search(query_vector, k=3)

for i, (score, idx) in enumerate(zip(scores[0], indices[0]), 1):
    if idx == -1:
        continue
    meta = all_metadata[idx]
    print(f"[Resultado {i}] Score L2: {score:.4f}")
    print(f"  Fuente:    {meta['fuente']}, pag. {meta['pagina']}")
    print(f"  Categoria: {meta['categoria']}")
    print(f"  Texto:     {meta['texto'][:200]}...")
    print()